# Join Strategies: Broadcast vs Sort-Merge vs Shuffle-Hash

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

For each strategy: **form a hypothesis before running the cell**, run it, read `.explain(mode="formatted")` yourself, *then* call `playbook.checkpoint(df, topic="join-strategies")` and click **Reveal self-check** on the topic page to compare against the manifest-driven annotation (US-2.1).

In [ ]:
import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("join-strategies").getOrCreate()
print("Spark version:", spark.version)
print("spark.sql.autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

## 1. Broadcast hash join

`autoBroadcastJoinThreshold` defaults to 10MB. Join a small dimension table (well under the threshold) against a larger fact table.

**Hypothesis:** which side gets broadcast? Do you expect an `Exchange` node for the large side at all?

In [ ]:
SCRATCH_DIR = "/workspace/scratch/join-strategies"

SMALL_ROWS = 2_000       # a few thousand rows, well under the 10MB broadcast threshold
LARGE_ROWS = 3_000_000
NUM_KEYS = SMALL_ROWS

small_df = spark.createDataFrame(
    [Row(id=i, category=f"cat-{i % 20}") for i in range(SMALL_ROWS)]
)

large_rdd = spark.sparkContext.parallelize(range(LARGE_ROWS), 24).map(
    lambda i: Row(id=i % NUM_KEYS, amount=float(i % 997))
)
large_df = spark.createDataFrame(large_rdd)

# A DataFrame built from an in-memory collection/RDD (createDataFrame()) has no
# size statistics Catalyst can use for the broadcast decision -- its estimated
# size defaults to spark.sql.defaultSizeInBytes (i.e. "unknown, assume large"),
# so even a genuinely tiny table like small_df would NOT auto-broadcast if joined
# directly (verified against a real cluster while building this notebook -- it
# silently fell back to a sort-merge join). Writing to parquet and reading back
# gives Spark a real file-size-based estimate, the way it would have for any real
# table -- this is what makes the broadcast decision below genuine, not assumed.
small_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/small")
large_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/large")
small_df = spark.read.parquet(f"{SCRATCH_DIR}/small")
large_df = spark.read.parquet(f"{SCRATCH_DIR}/large")

print("small_df rows:", small_df.count(), " large_df rows:", large_df.count())

In [ ]:
broadcast_join = large_df.join(small_df, on="id", how="inner")
broadcast_join.count()  # materialize
broadcast_join.explain(mode="formatted")

In [ ]:
checkpoint(broadcast_join, topic="join-strategies")
print("Checkpoint written -- now click 'Reveal self-check' on the topic page.")

## 2. Sort-merge join

Disable broadcasting entirely (`autoBroadcastJoinThreshold = -1`) and join two large tables. Neither side is small enough (nor is broadcast even allowed) to hash-broadcast, so Spark falls back to a sort-merge join: both sides shuffled (`Exchange`) and sorted (`Sort`) on the join key before merging.

**Hypothesis:** how many `Exchange` nodes do you expect this time, and where?

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

LARGE_B_ROWS = 3_000_000
large_b_rdd = spark.sparkContext.parallelize(range(LARGE_B_ROWS), 24).map(
    lambda i: Row(id=i % NUM_KEYS, region=f"region-{i % 8}")
)
large_b_df = spark.createDataFrame(large_b_rdd)
large_b_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/large_b")
large_b_df = spark.read.parquet(f"{SCRATCH_DIR}/large_b")

sort_merge_join = large_df.join(large_b_df, on="id", how="inner")
sort_merge_join.count()
sort_merge_join.explain(mode="formatted")

In [ ]:
checkpoint(sort_merge_join, topic="join-strategies")
print("Checkpoint written -- Reveal again to compare against this plan.")

## 3. Shuffle-hash join (forced via config)

Set `spark.sql.join.preferSortMergeJoin=false` and pick an `autoBroadcastJoinThreshold` deliberately *below* the medium table's real size but *above* zero. This matters for a subtle, real reason found while building this notebook: Spark's shuffle-hash-join eligibility check (`canBuildLocalHashMap`) reuses the *same* `autoBroadcastJoinThreshold` config, multiplied by `spark.sql.shuffle.partitions` -- **setting the threshold to `-1` (as step 2 did, to force sort-merge) disables shuffle-hash-join eligibility too**, not just broadcast. A small positive threshold is required here, not a fully-disabled one, or Spark can never pick `ShuffledHashJoin` no matter what `preferSortMergeJoin` says.

**Hypothesis:** do you expect a `Sort` node on the smaller side this time?

In [ ]:
spark.conf.set("spark.sql.join.preferSortMergeJoin", "false")
# Small enough that medium_df's real (post-parquet) size exceeds it (won't just broadcast), but leaves
# plenty of room under threshold * spark.sql.shuffle.partitions (this cluster's 200) for medium_df to
# still qualify as a shuffle-hash build side.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 20 * 1024)

MEDIUM_ROWS = 100_000  # small next to large_df, but its real parquet size clears the 20KB threshold above
medium_rdd = spark.sparkContext.parallelize(range(MEDIUM_ROWS), 8).map(
    lambda i: Row(id=i % NUM_KEYS, tag=f"tag-{i % 5}")
)
medium_df = spark.createDataFrame(medium_rdd)
medium_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/medium")
medium_df = spark.read.parquet(f"{SCRATCH_DIR}/medium")

shuffle_hash_join = large_df.join(medium_df, on="id", how="inner")
shuffle_hash_join.count()
shuffle_hash_join.explain(mode="formatted")

In [ ]:
checkpoint(shuffle_hash_join, topic="join-strategies")
print("Checkpoint written -- Reveal again. If the plan still shows SortMergeJoin, Spark's cost "
      "heuristic decided the shuffle-hash path wasn't cheap enough here -- that's a real, teachable "
      "outcome, not a notebook bug; try shrinking MEDIUM_ROWS further and re-run this cell.")

## 4. Reset session-level confs

Reset the confs this notebook changed so a later run (or another topic reusing this session) isn't affected.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")